# Assignment 3: Fine-tuning language models

In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM on an instruction tuning dataset. You will convert this dataset into instruction-response pairs, fine-tune a causal language model using LoRA (Low-Rank Adaptation), and evaluate it through prompted inference and comparison with other methods.

## Preliminaries

First, let's install the required libraries. If you are running in your own environment, make sure the following are installed:

- [Torch](https://docs.pytorch.org/docs/stable/index.html)
- [Transformers](https://huggingface.co/docs/transformers/index)
- [Datasets](https://huggingface.co/docs/datasets/index)
- [Evaluate](https://huggingface.co/docs/evaluate/en/index)
- [NLTK](https://www.nltk.org/api/nltk.html)
- [rouge_score](https://pypi.org/project/rouge-score/)

In a Colab notebook, most of them are already installed, except Evaluate and rouge_score.

In [1]:
%pip install evaluate rouge_score

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


Note: you may need to restart the kernel to use updated packages.


We also set some configuration parameters.

Most importantly, you should select a language model to work with in this assignment and enter its HuggingFace identifier in the parameter `MODEL_NAME` below. In principle you can use any model that you want, but we recommend that you select a model that has not already been trained to follow instructions, so it should be a "pure" language model trained on raw text (similar to Assignments 1 and 2).

The selected model should be small enough to fit in your computational environment. We have verified that the 135-million parameter [`SmolLM2` model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), developed by HuggingFace, can be used to solve this assignment in a Colab notebook (free tier, T4 GPU). If you run on a cluster, you can select a larger model (and probably see more interesting results).

We also define training and test set sizes here. Again, the values below have been set so that the assignment can be solved in Colab, and you can increase these sizes to improve the quality of the fine-tuned models.

In [1]:
import torch
import torch.nn as nn

SEED = 101
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_TRAIN_SAMPLES = 5000
MAX_TEST_SAMPLES = 400

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"

/home/hoda/anaconda3/envs/tch/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


# Part 1: Preprocessing

### ⚙&nbsp; Task 1.1: Loading and inspecting the dataset

The dataset [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) is a collection of instruction-response pairs designed for SFT of large language models for instruction following. This dataset consists of examples of user inputs with system responses.

You can load using the datasets from the HuggingFace repository as follows.

In [2]:
from datasets import load_dataset
from datasets import DatasetDict

smoltalk = load_dataset("HuggingFaceTB/smoltalk", 'all')

In order to make this assignment possible to solve in a restricted environment, we simplify the dataset a bit:
- We remove multi-turn chat dialogues from the dataset;
- We remove instances where the query or the answer is greater than a set maximum length;
- We keep a subset of the data for training and testing (by default 5000 and 400, respectively).

In [3]:
smoltalk_simplified = smoltalk.filter(lambda row: len(row['messages']) <= 3 and all(len(m['content']) <= 256 for m in row['messages']))
smoltalk_simplified = DatasetDict({
    "train": smoltalk_simplified["train"].select(range(MAX_TRAIN_SAMPLES)),
    "test": smoltalk_simplified["test"].select(range(MAX_TEST_SAMPLES)),
})

In [5]:
smoltalk_simplified

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 400
    })
})

Print some examples from the dataset so that you understand the format.

Key points you need to note here: each example from the training or test set consists of a sequence of messages. The number of messages in each example will be 2 or 3, because we removed multi-turn chat dialogues in the previous step. Each message is associated with a `role` label:
- `user`: an example of something the user might write.
- `assistant`: an example of an output an LLM could be expected to produce, given the input.
- `system`: a *system prompt* that gives guidelines for the general behavior of the LLM's behavior.

All examples in the dataset include a user input and an assistant output, but the system prompt is not available in all of the examples.

In [6]:
smoltalk_simplified['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting'}

### 🎓&nbsp; Task 1.2: Formatting the data for instruction tuning

Define a function `format_input_output` that converts an example from the dataset into an input/output pair that we can use to fine-tune the LLM.

You are free to design the format. The following document gives some examples that have been used by different instruction-following LLMs including Llama and Mistral: https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats

The later stages of our preprocessing pipeline expect that this function returns an object containing two parts: the `prompt` (what goes into the LLM before generating anything) and the `response` (what the LLM is expected to generate).

In [4]:
def format_input_output(example):
  # `messages` is a list of messages, each with a `content` string and a `role`.
  messages = example['messages']

  # TODO: implement this — return {"prompt": ..., "response": ...}
  system_msg = ""
  user_msg = ""
  assistant_msg = ""

  for m in messages:
    if m['role'] == 'system':
      system_msg = m['content']
    elif m['role'] == 'user':
      user_msg = m['content']
    elif m['role'] == 'assistant':
      assistant_msg = m['content']

  # Build the prompt using ChatML format (used by SmolLM2)
  prompt = ""
  if system_msg:
    prompt += f"<|im_start|>system\n{system_msg}<|im_end|>\n"
  prompt += f"<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n"

  response = f"{assistant_msg}<|im_end|>"

  return {"prompt": prompt, "response": response}

Apply the function you implemented to the dataset as a whole.

In [5]:
ds_sft = smoltalk_simplified.map(format_input_output)

Then verify that the dataset now contains the new fields you created.

In [6]:
ds_sft['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting',
 'prompt': "<|im_start|>system\nYou are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.<|im_end|>\n<|im_start|>user\nRearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.<|im_end|>\n<|im_start|>assistant\n",
 'response': 'The chef made more food after the restaurant ran out.<|im_end|>'}

### ⚙&nbsp; Task 1.3: Tokenizing the dataset

We will now prepare the format required by the HuggingFace Trainer.

We first load the tokenizer for our selected model:

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Write a function `tokenize_helper` that takes an example (using the prompt/response format from the previous step) and produces the following three results:

- `input_ids`: the integer token ids of the concatenated prompt and response;
- `labels`: a list of the same length as `input_ids`, where the response token ids are the same, but where the prompt token ids have all been replaced by the loss masking identifier -100.
- `attention_mask`: the attention mask. This should just be a list of the same length as the other two lists, with all items set to 1.

The reason why `input_ids` and `labels` are different is that
we do not want to compute the training loss for tokens that appear in the user's input. We want to train the model to generate output *conditionally*: based on a prompt. But why the magic number -100? This is the number used by default in PyTorch's [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) to indicate an item that should be excluded in loss computations. (This issue was also mentioned in [Assignment 1](https://liu-nlp.ai/dl4nlp/units/a1_1.html#task-4.1-implementing-the-trainer).)

In [ ]:
def tokenize_helper(example):
    prompt = example['prompt']     
    response = example['response'] # Created in the previous step

    # TODO: produce input_ids, attention_mask, and labels.
    #       labels should mask prompt tokens with -100 so loss is only on the response.
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    response_ids = tokenizer(response, add_special_tokens=False)['input_ids']

    input_ids = prompt_ids + response_ids #join them into one sequence: [prompt tokens] + [response tokens]
    attention_mask = [1] * len(input_ids)
    # Mask prompt tokens with -100 so loss is only computed on the response
    labels = [-100] * len(prompt_ids) + response_ids

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


As above, apply the function you implemented to the dataset using `map`. This will add the three new fields to the dataset.

In [12]:
tokenized_ds_sft = ds_sft.map(tokenize_helper)
tokenized_ds_sft

DatasetDict({
    train: Dataset({
        features: ['messages', 'source', 'prompt', 'response', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source', 'prompt', 'response', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 400
    })
})


## Part 2: Evaluation of the baseline model

As a first step, we will see how well the *baseline* model performs: that is, a model that has not been trained to follow instructions.

### ⚙&nbsp; Task 2.1: Preparing for evaluation

In this section, we set up a few utilities we will need to complete our training and evaluation infrastructure. These utilities will be given and you don't need to modify anything.

The first piece we need is a *collator*: that is, a tool that takes a number of instances and creates PyTorch tensors for a training batch. To make the batch fit into rectangular tensors, padding tokens will be added.

In [8]:
def data_collator(batch):
    """
    Create a custom collate function for causal language modeling.

    Args:
        batch: List of examples, each with 'input_ids', 'attention_mask', 'labels'
        tokenizer: Tokenizer with pad_token_id
    """

    input_ids_list = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    attention_masks_list = [torch.tensor(example["attention_mask"], dtype=torch.long) for example in batch]
    labels_list = [torch.tensor(example['labels'], dtype=torch.long) for example in batch]

    # Find max length in this batch
    max_len = max(x.size(0) for x in input_ids_list)

    # Helper pad function
    def pad_to_max(x_list, pad_value):
        padded = []
        for x in x_list:
            pad_len = max_len - x.size(0)
            if pad_len > 0:
                pad_tensor = torch.full((pad_len,), pad_value, dtype=x.dtype)
                x = torch.cat([x, pad_tensor], dim=0)
            padded.append(x)
        return torch.stack(padded, dim=0)

    # Use tokenizer.pad_token_id for inputs, 0 for attention_mask, -100 for labels
    pad_id = tokenizer.pad_token_id

    batch_input_ids = pad_to_max(input_ids_list, pad_value=pad_id)
    batch_attention_mask = pad_to_max(attention_masks_list, pad_value=0)
    batch_labels = pad_to_max(labels_list, pad_value=-100)

    batch = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }
    return batch

The second utility we need is an evaluator. We will use the **ROUGE-L** metric, which computes the longest common subsequence between the model's output and the gold-standard answer. You can read about ROUGE-L here: https://en.wikipedia.org/wiki/ROUGE_(metric)

When using the ROUGE-L metric in a Trainer, we need to wrap it in an object defined as follows:

In [9]:
import evaluate

class RougeMetricComputer:
    """
    Stateful metric for batch_eval_metrics=True.

    It:
      - accumulates predictions and references across batches
      - computes ROUGE-L once at the end (compute_result=True)
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.rouge = evaluate.load("rouge")
        self.all_predictions = []
        self.all_references = []

    def __call__(self, eval_pred, compute_result=False):
        """Accumulate predictions and compute at the end."""

        logits, labels = eval_pred
        pred_ids = logits.argmax(axis=-1)

        # Collect decoded answer-span text from each example in the batch
        for p, lbl in zip(pred_ids, labels):
            mask = lbl != -100
            if mask.sum() == 0:
                continue

            ref_ids = lbl[mask]
            pred_ids_filtered = p[mask]

            ref_text = self.tokenizer.decode(ref_ids, skip_special_tokens=True)
            pred_text = self.tokenizer.decode(
                pred_ids_filtered, skip_special_tokens=True,
                eos_token_id=self.tokenizer.vocab['<|im_end|>']
            )

            self.all_references.append(ref_text.strip())
            self.all_predictions.append(pred_text.strip())

        # Only compute at the very end of eval
        if compute_result:
            if len(self.all_references) > 0:
                scores = self.rouge.compute(
                    predictions=self.all_predictions,
                    references=self.all_references,
                )

                # Clear accumulated data for next eval call
                self.all_predictions = []
                self.all_references = []
                return {"rougeL": scores["rougeL"]}
            else:
                return {}
        else:
            return {}

compute_metrics = RougeMetricComputer(tokenizer)


2026-06-11 15:44:46.261590: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Finally, we make a function that sets up a [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer).

In [15]:
from transformers import Trainer
from transformers.trainer_callback import ProgressCallback

def make_trainer(model, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds_sft["train"],
        eval_dataset=tokenized_ds_sft["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if type(cb).__name__ != "NotebookProgressCallback"
    ]
    trainer.add_callback(ProgressCallback)
    return trainer


### 🎓&nbsp; Task 2.2: Evaluating the pre-trained model

Now, we have all the pieces to evaluate our baseline model that has not been instruction-tuned.

The following code will compute the loss on the test set as well as the ROUGE-L score. You will later compare these scores to the models that you train.

Why do you think the ROUGE-L score is as high as it is, even without any training for instruction-following?

> **Answer:** The 0.57 score surprised me. I think it is high because ROUGE-L counts word overlap, and the model already knows English well, so it naturally uses many of the same common words as the references. For the grammar task it sometimes shows part of put which still overlaps with the gold answer. So the score reflects language fluency more than actually following instructions.

In [16]:
import json
from transformers import TrainingArguments
from transformers import AutoModelForCausalLM
import time

print("\n" + "=" * 80)
print("EVALUATING PRETRAINED MODEL")
print("=" * 80)

pretrained_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

pretrained_eval_args = TrainingArguments(
    output_dir="/tmp/pretrained_eval",
    eval_strategy="no",
    per_device_eval_batch_size=1,
    bf16=True, fp16=False,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

pretrained_trainer = make_trainer(pretrained_model, pretrained_eval_args)

t0 = time.perf_counter()
pretrained_eval_metrics = pretrained_trainer.evaluate()
pretrained_eval_time = time.perf_counter() - t0

print("\nPRETRAINED EVAL METRICS:")
print(json.dumps(pretrained_eval_metrics, indent=2))


EVALUATING PRETRAINED MODEL


  0%|          | 0/400 [00:00<?, ?it/s]


PRETRAINED EVAL METRICS:
{
  "eval_loss": 2.6185266971588135,
  "eval_model_preparation_time": 0.0023,
  "eval_rougeL": 0.5731055740126778,
  "eval_runtime": 20.0008,
  "eval_samples_per_second": 19.999,
  "eval_steps_per_second": 19.999
}



## Part 3: Supervised fine-tuning



### 🎓&nbsp; Task 3.1: Training the full model

Next, we train the pre-trained model using SFT over all the parameters, then calculate the metrics and outputs to evaluate how well it follows instructions.

How do the results differ from those in the previous step?

> **Answer:** The improvement is clear. Loss went from 2.62 to 1.15 and ROUGE-L from 0.57 to 0.67. More importantly the actual outputs changed. The pretrained model just repeats the question, but after SFT it gives real answers. For the grammar task it outputs exactly the corrected sentence. One epoch was enough for the model to learn the answer format.

In [17]:
import time

baseline_training_args = TrainingArguments(
    output_dir="/tmp/baseline_trainer",
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=True, fp16=False,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
baseline_trainer = make_trainer(base_model, baseline_training_args)

print("\n" + "=" * 80)
print("TRAINING FULL MODEL (SFT)")
print("=" * 80)

sft_t0 = time.perf_counter()
baseline_trainer.train()
sft_train_time = time.perf_counter() - sft_t0

print("\n" + "=" * 80)
print("EVALUATING FULL SFT MODEL")
print("=" * 80)

baseline_eval_metrics = baseline_trainer.evaluate()
sft_loss   = baseline_eval_metrics["eval_loss"]
sft_rougeL = baseline_eval_metrics.get("eval_rougeL", float("nan"))

print(f"\nFull SFT  |  train time: {sft_train_time:.1f}s  |  loss: {sft_loss:.4f}  |  ROUGE-L: {sft_rougeL:.4f}")
print(json.dumps(baseline_eval_metrics, indent=2))


TRAINING FULL MODEL (SFT)


  0%|          | 0/5000 [00:00<?, ?it/s]

{'loss': 1.393, 'grad_norm': 1.3405046463012695, 'learning_rate': 3.001e-05, 'epoch': 0.4}


{'loss': 1.1955, 'grad_norm': 5.948265075683594, 'learning_rate': 1.001e-05, 'epoch': 0.8}


  0%|          | 0/400 [00:00<?, ?it/s]

{'eval_loss': 1.1548659801483154, 'eval_rougeL': 0.6715426556326718, 'eval_runtime': 16.4773, 'eval_samples_per_second': 24.276, 'eval_steps_per_second': 24.276, 'epoch': 1.0}
{'train_runtime': 393.4814, 'train_samples_per_second': 12.707, 'train_steps_per_second': 12.707, 'train_loss': 1.25994599609375, 'epoch': 1.0}

EVALUATING FULL SFT MODEL


  0%|          | 0/400 [00:00<?, ?it/s]


Full SFT  |  train time: 393.7s  |  loss: 1.1549  |  ROUGE-L: 0.6715
{
  "eval_loss": 1.1548659801483154,
  "eval_rougeL": 0.6715426556326718,
  "eval_runtime": 16.7528,
  "eval_samples_per_second": 23.877,
  "eval_steps_per_second": 23.877,
  "epoch": 1.0
}


### ⚙&nbsp; Task 3.3: Counting the number of trainable parameters

Define a function `num_trainable_parameters` that computes the number of floating-point numbers that a given model will update during training.

**Hints**:
- For a PyTorch module `m`, you can use `m.parameters()` to access its parameter tensors.
- However, you should only include parameter tensors where the flag `requires_grad` is True.


In [18]:
def num_trainable_parameters(model):
    """Count number of trainable parameters.

    Args:
        model: A PyTorch module.
    """
    # TODO: sum numel() for parameters where requires_grad is True
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

Apply this function to the SFT-trained model and check that the result makes sense.

In [19]:
n_params_full = num_trainable_parameters(base_model)
print(f"Full SFT model trainable parameters: {n_params_full:,}")

Full SFT model trainable parameters: 134,515,008


## Part 4: Parameter-efficient fine-tuning

In the last section of this assignment, we will use LoRA to train the model in a more parameter-efficient manner. You may want to prepare by reading  by [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685) and the teaching material provided for this course.

### ⚙&nbsp; Task 4.1: Utilities for modifying models

Define a function `extract_lora_targets` that extracts the relevant linear layers from all Transformer blocks in your selected LLM.
It is up to you to decide what layers to select; in the experiments described in the original LoRA paper, the query and value projection matrices were fine-tuned with LoRA, while all other layers were left unchanged.
Return a dictionary that maps the component name to the corresponding linear layer.

As we saw earlier (in Assignment 2 and elsewhere), a Transformer model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. You can use get_submodule() to retrieve a layer by a string name. This name depends on the model you have selected. For instance, in the `SmolLM2-135M` model, `'model.layers.0.self_attn.q_proj'`
 refers to the query projection in Transformer layer 0.

It is OK to hard-code this part, so that you just enumerate the layers you want to extract. Alternatively, use a utility such as `model.named_modules()` to iterate through the model's layers.

In [20]:
def extract_lora_targets(model):
    # TODO: return a dict mapping fully-qualified layer names to the
    #       linear layers in attention blocks (q_proj, k_proj, v_proj, o_proj).
    targets = {}
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            if any(proj in name for proj in ['q_proj', 'k_proj', 'v_proj', 'o_proj']):
                targets[name] = module
    return targets

We also need a convenience function that puts layers back into a model. The following function does the trick. The `named_layers` argument uses the same format as returned by `extract_lora_targets`.

In [21]:
def replace_layers(model, named_layers):
    """
    Replace submodules in `model` by name.
    """
    for name, layer in named_layers.items():
        components = name.split(".")
        submodule = model
        for comp in components[:-1]:
            submodule = getattr(submodule, comp)
        setattr(submodule, components[-1], layer)
    return model

### 🎓&nbsp; Task 4.2: Implementing the LoRA layer

To implement the LoRA approach, we define a new type of layer that will be used as a drop-in replacement for a regular linear layer.

In [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685), the structure is presented visually in Figure 1, and equation (3) shows the same idea.

Start from the following skeleton and fill in the missing pieces:


In [22]:
import torch.nn as nn

class LoRALayer(nn.Module):
    def __init__(self, W, r, alpha):
        super().__init__()
        # TODO: freeze W, initialise A (random Gaussian) and B (zeros)
        self.W = W
        for param in self.W.parameters():
            param.requires_grad = False

        d_out = W.out_features
        d_in = W.in_features

        # A initialized with random Gaussian (scaled), B initialized to zero
        # This ensures ΔW = BA = 0 at the start of training (Hu et al. 2021, Sec 4.1)
        self.A = nn.Parameter(torch.randn(r, d_in) / (r ** 0.5))
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.r = r
        self.alpha = alpha

    def forward(self, x):
        # TODO: h = W₀x + (α/r) * B * A * x  (Hu et al. 2021, Eq. 3)
        lora_update = (x @ self.A.T) @ self.B.T
        return self.W(x) + (self.alpha / self.r) * lora_update

Here, `W` is the linear layer we are fine-tuning, while `r` and `alpha` are hyperparameters described in section 4.1. of the paper. The `r` parameter controls the parameter efficiency: by setting it to a low value, we save memory but make a rougher approximation. The `alpha` parameter is a scaling factor.

### 🎓&nbsp; Task 4.3: Fine-tuning with LoRA

Set up a model where you replace the four linear layers in attention blocks (query, key, value, and output) with LoRA layers. Use the following steps:
- First use `extract_lora_targets` to get the relevant linear layers.
- Each of the linear layers in the returned dictionary should be wrapped inside a LoRA layer.
- Then use `replace_layers` to put them back into the model.

Train this model and compare the training speed, metrics, and outputs to the results from Part 3.

Apply your parameter counting function (`num_trainable_parameters`) to this model, compare the results to those in Part 3, and make sure that these results correspond to your expectations.

> **Answer:** LoRA trains only 921,600 parameters vs 134,515,008 for full SFT, which is 146x fewer. Metrics are a bit worse (loss 1.46 vs 1.15, ROUGE-L 0.65 vs 0.67) which makes sense with fewer parameters. The surprising thing is LoRA was actually slower (480s vs 394s). With batch size 1 there is no real memory saving, and the extra matrix operations add overhead. With larger batch sizes LoRA would probably be faster.

In [23]:
LORA_R = 8
LORA_ALPHA = 16

# Load fresh model and freeze all parameters
lora_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
for param in lora_model.parameters():
    param.requires_grad = False

# Replace attention projection layers with LoRA layers
lora_targets = extract_lora_targets(lora_model)
lora_layers = {name: LoRALayer(layer, r=LORA_R, alpha=LORA_ALPHA).to(DEVICE)
               for name, layer in lora_targets.items()}
lora_model = replace_layers(lora_model, lora_layers)

n_params_lora = num_trainable_parameters(lora_model)
print(f"LoRA trainable parameters:     {n_params_lora:,}")
print(f"Full SFT trainable parameters: {n_params_full:,}")
print(f"Parameter reduction:           {n_params_full / n_params_lora:.1f}x fewer")

lora_training_args = TrainingArguments(
    output_dir="/tmp/lora_trainer",
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=True, fp16=False,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

lora_trainer = make_trainer(lora_model, lora_training_args)

print("\n" + "=" * 80)
print("TRAINING LORA MODEL")
print("=" * 80)

lora_t0 = time.perf_counter()
lora_trainer.train()
lora_train_time = time.perf_counter() - lora_t0

print("\n" + "=" * 80)
print("EVALUATING LORA MODEL")
print("=" * 80)

lora_eval_metrics = lora_trainer.evaluate()
lora_loss   = lora_eval_metrics["eval_loss"]
lora_rougeL = lora_eval_metrics.get("eval_rougeL", float("nan"))

print(f"\nLoRA      |  train time: {lora_train_time:.1f}s  |  loss: {lora_loss:.4f}  |  ROUGE-L: {lora_rougeL:.4f}")
print(json.dumps(lora_eval_metrics, indent=2))

LoRA trainable parameters:     921,600
Full SFT trainable parameters: 134,515,008
Parameter reduction:           146.0x fewer

TRAINING LORA MODEL


  0%|          | 0/5000 [00:00<?, ?it/s]

{'loss': 1.6802, 'grad_norm': 3.819288730621338, 'learning_rate': 3.001e-05, 'epoch': 0.4}


{'loss': 1.4661, 'grad_norm': 8.75454044342041, 'learning_rate': 1.001e-05, 'epoch': 0.8}


  0%|          | 0/400 [00:00<?, ?it/s]

{'eval_loss': 1.4614337682724, 'eval_rougeL': 0.6454083929314199, 'eval_runtime': 19.2676, 'eval_samples_per_second': 20.76, 'eval_steps_per_second': 20.76, 'epoch': 1.0}
{'train_runtime': 480.2197, 'train_samples_per_second': 10.412, 'train_steps_per_second': 10.412, 'train_loss': 1.5364952392578124, 'epoch': 1.0}

EVALUATING LORA MODEL


  0%|          | 0/400 [00:00<?, ?it/s]


LoRA      |  train time: 480.5s  |  loss: 1.4614  |  ROUGE-L: 0.6454
{
  "eval_loss": 1.4614337682724,
  "eval_rougeL": 0.6454083929314199,
  "eval_runtime": 19.1271,
  "eval_samples_per_second": 20.913,
  "eval_steps_per_second": 20.913,
  "epoch": 1.0
}


In [24]:
# ── Comparison Summary ──────────────────────────────────────────────────────

pretrained_loss   = pretrained_eval_metrics["eval_loss"]
pretrained_rougeL = pretrained_eval_metrics.get("eval_rougeL", float("nan"))

col = "{:<22}"
num = "{:>14}"
print("\n" + "=" * 72)
print(f"{'Model':<22} {'Train time (s)':>14} {'# Params':>14} {'Eval loss':>10} {'ROUGE-L':>8}")
print("-" * 72)
print(f"{'Pretrained (no SFT)':<22} {'—':>14} {n_params_full:>14,} {pretrained_loss:>10.4f} {pretrained_rougeL:>8.4f}")
print(f"{'Full SFT':<22} {sft_train_time:>14.1f} {n_params_full:>14,} {sft_loss:>10.4f} {sft_rougeL:>8.4f}")
print(f"{'LoRA SFT':<22} {lora_train_time:>14.1f} {n_params_lora:>14,} {lora_loss:>10.4f} {lora_rougeL:>8.4f}")
print("=" * 72)
print(f"\nLoRA is {sft_train_time / lora_train_time:.1f}x faster than full SFT")
print(f"LoRA uses {n_params_full / n_params_lora:.1f}x fewer trainable parameters")


Model                  Train time (s)       # Params  Eval loss  ROUGE-L
------------------------------------------------------------------------
Pretrained (no SFT)                 —    134,515,008     2.6185   0.5731
Full SFT                        393.7    134,515,008     1.1549   0.6715
LoRA SFT                        480.5        921,600     1.4614   0.6454

LoRA is 0.8x faster than full SFT
LoRA uses 146.0x fewer trainable parameters


### 🎓&nbsp; Task 4.4: Qualitative inspection

Run the three models interactively on some examples of your own choice (either taken from the training or test sets, or created by yourself). The convenience function below can be of use, but you need to complete it by using the prompt format you defined in Task 1.2.

Do your models seem to have learned the instruction-following behavior (at least to some extent)? Do they respond to user queries sensibly?

The quality we see here will depend on your choice of base model as well as how much you trained it.

> **Answer:** Yes both fine-tuned models learned to follow instructions. The pretrained model just repeats the question or keeps outputting "Assistant" which is useless. Both SFT and LoRA correctly answer "The capital of France is Paris" and both fix the grammar task properly. The models still repeat themselves a lot though and do not always stop at the right point. LoRA's poem output was mostly the same sentence reworded rather than actual verse, while SFT at least produced something that looked like a poem. Both results are reasonable given only 1 epoch on a 135M parameter model.

In [25]:
def generate_response(model, user_text, max_new_tokens=128):
    """Generate a response from a model given a plain user message."""
    example = {"messages": [{"role": "user", "content": user_text}]}
    prompt = format_input_output(example)["prompt"]

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# Test prompts: a general question, a task-style instruction, and one from the dataset
# Use role-based lookup so it works for both 2-message [user, assistant]
# and 3-message [system, user, assistant] examples
test_example_user = next(
    m['content'] for m in ds_sft['test'][0]['messages'] if m['role'] == 'user'
)
test_prompts = [
    "What is the capital of France?",
    "Write a short poem about the sea.",
    test_example_user,
]

models = {
    "Pretrained (no fine-tuning)": pretrained_model,
    "Full SFT": base_model,
    "LoRA SFT": lora_model,
}

for user_text in test_prompts:
    print("\n" + "=" * 70)
    print(f"USER: {user_text}")
    for label, model in models.items():
        response = generate_response(model, user_text)
        print(f"\n[{label}]\n{response}")


USER: What is the capital of France?



[Pretrained (no fine-tuning)]
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?




[Full SFT]
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital



[LoRA SFT]
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital of France is Paris.
The capital

USER: Write a short poem about the sea.



[Pretrained (no fine-tuning)]
Write a short poem about the sea.
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant



[Full SFT]
The sea whispers secrets,
The waves dance in the sun,
The sun shines on the sea,
The sea whispers secrets.
The sea whispers secrets.
The sea whispers secrets.
The sea whispers secrets.
The sea whispers secrets.
The sea whispers secrets.
The sea whispers secrets.
The sea whispers secrets.
The sea whispers secrets.
The sea whispers secrets.
The sea whispers secrets.
The sea whispers secrets.
The sea whispers secrets.
The sea whispers secrets.
The sea whispers secrets.
The



[LoRA SFT]
The sea is a vast expanse of water, stretching out as far as the eye can see. It is a place where the sun sets and the moon rises, where the waves crash against the shore, and where the sea is a place of mystery and wonder. It is a place where the sea is a place of mystery and wonder.

The sea is a place where the sun sets and the moon rises, where the waves crash against the shore, and where the sea is a place of mystery and wonder.

The sea is a place where the sun sets and the moon rises, where the waves crash against

USER: Correct the verb tense in the following sentence: "I swim every day last week.":
"I swam every day last week."



[Pretrained (no fine-tuning)]
Correct the verb tense in the following sentence: "I swim every day last week.":
"I swam every day last week."

Correct the verb tense in the following sentence: "I swim every day last week.":
"I swam every day last week."

Correct the verb tense in the following sentence: "I swim every day last week.":
"I swam every day last week."

Correct the verb tense in the following sentence: "I swim every day last week.":
"I swam every day last week."

Correct the verb tense in the following sentence: "I swim



[Full SFT]
"I swam every day last week."



[LoRA SFT]
"I swam every day last week."
"I swam every day last week."
"I swam every day last week."

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           

        

           
